# `adrqa` API: Building a Decision Graph From Architecture Decision Records

- This notebook walks through the offline API of `adrqa` one building block
  at a time, on the decision records (ADRs) of `eclipse-edc/Connector` at one
  pinned commit, with no API key
- The pedagogical arc:
  - Read the repository at a pinned commit without checking it out
  - Parse one record into fields, references and replacements
  - Check a language model's replacement claims against the record text
  - Link the record to the code it governs
  - Build the decision graph and query it
  - Score staleness and rank documentation gaps

In [1]:
!pip install -q --root-user-action=ignore lizard==1.25.0 markdown-it-py==4.0.0 networkx==3.5 python-dotenv==1.2.4

import lizard
import markdown_it
import networkx

print("lizard version: ", lizard.version)
print("markdown-it-py version: ", markdown_it.__version__)
print("networkx version: ", networkx.__version__)

lizard version:  1.25.0
markdown-it-py version:  4.0.0
networkx version:  3.5


In [2]:
%load_ext autoreload
%autoreload 2

import logging

import pandas as pd

In [3]:
import helpers.hnotebook as hnotebook

import UmdTask745_MSML610_Fall2026_AI_Knowledge_Management_Development_utils as utils

# Initialize notebook configuration and logging.
hnotebook.config_notebook()
_LOG = logging.getLogger(__name__)
utils.init_loggers(_LOG)

# Make the `adrqa` package under `src/` importable.
utils.add_src_to_path()

# Convert `display` into `print()` when running outside IPython.
try:
    from IPython.display import display
except ImportError:
    display = print  # type: ignore

INFO  > /usr/local/lib/python3.12/site-packages/ipykernel_launcher.py -f /tmp/tmpbbu4uml0.json --HistoryManager.hist_file=:memory:


In [4]:
import adrqa.config as aconfig
import adrqa.gaps as agaps
import adrqa.gitstore as agitsto
import adrqa.graph as agraph
import adrqa.linker as alinker
import adrqa.parser as aparser
import adrqa.relations as arelati
import adrqa.staleness as astalen

## Library Overview

- **What problem it solves**: a repository keeps its design decisions in
  Markdown records, but nothing ties a record to the code it governs, to the
  record that replaced it, or says whether it still matches the code
- **Key abstraction**: a decision graph whose nodes are records and files,
  and whose edges are `governs`, `supersedes` and `references`
- **Mental model**:

| Object | Description | Output |
| :----- | :---------- | :----- |
| `GitStore` | Reads git objects at the pinned commit | file text, history |
| `parse_record()` | Markdown record to labelled pieces, no model | record dict |
| `model_edges()` | Optional model finds replacements, then checked | edges |
| `Linker` | Names in the text to files in the tree | link dicts |
| `build_graph()` | Records, files and the three edge kinds | `networkx` graph |
| `score_record()` | Later code commits / (1 + record edits) | staleness |
| `rank_gaps()` | Complexity x (1 - coverage) per module | gap list |

- The `adrqa build` command runs all of them in this order and writes the
  results to `data/`, see the example notebook

# Part 1: Read the Repository at a Pinned Commit

## Cell 1.1: Clone Once and Pin the Commit

- The corpus is fixed: one repository, one commit SHA, ten records, so every
  run reads exactly the same files
- `GitStore` never checks files out: it reads git objects with `git show`
  and `git cat-file`, which also avoids the Windows path length limit
- The first run clones the repository into `.cache/` (about 70 MB)

In [5]:
# Clone the repository if needed and open it at the pinned commit.
agitsto.GitStore.ensure_clone()
store = agitsto.GitStore()
print("repository:", aconfig.REPO_URL)
print("pinned commit:", store.sha[:9])
print("paths in the tree:", len(store.tree))
print("records used:", len(aconfig.RECORDS))

repository: https://github.com/eclipse-edc/Connector.git
pinned commit: a66db8963
paths in the tree: 3534
records used: 10


## Cell 1.2: Read One Decision Record

- We follow one record through the whole notebook: the 2026-09-22 record on
  per-participant resource ids, because it partly replaces an older record

In [6]:
record_id = "2026-09-22-per-participant-resource-ids"
path = f"{aconfig.RECORD_ROOT}{record_id}/README.md"
text = store.read(path)
print("\n".join(text.splitlines()[:22]))

# Per-participant resource identifiers

## Decision

The identity of an `Asset`, a `ContractDefinition` and a `PolicyDefinition` is the pair
`(participantContextId, id)`. Their ids are unique only within the owning participant context, not across the whole
connector:

- the SQL primary keys of `edc_asset`, `edc_contract_definitions` and `edc_policydefinitions` are composite keys on
  `participant_context_id` and the resource id, and the in-memory stores are keyed the same way;
- the store SPIs (`AssetIndex`, `ContractDefinitionStore`, `PolicyDefinitionStore`) and the service SPIs
  (`AssetService`, `ContractDefinitionService`, `PolicyDefinitionService`) resolve, update and delete these resources by
  participant context id **and** resource id. The methods that took only a resource id have been removed, there is no
  "global" lookup anymore;
- every code path that needs one of these resources passes the participant context it acts on behalf of: the Management
  API (`v4` from the runtim

# Part 2: Parse a Record Without a Model

## Cell 2.1: Fields, References and Replacements

- `parse_record()` uses `markdown-it-py` to find the title and cut the text
  at its `##` headings into Decision, Rationale and Approach
- Links of the form `../<record>/README.md` become references
- A sentence with "supersedes" and a link becomes a replacement edge; a
  quoted statement in that sentence makes it a partial replacement

In [7]:
corpus = set(aconfig.RECORDS)
record = aparser.parse_record(record_id, text, corpus)
print("title:", record["title"])
print("date:", record["date"])
print("status:", record["status"])
print("missing fields:", record["missing_fields"])
print("references:", record["references"])
display(pd.DataFrame(record["supersedes"]))

title: Per-participant resource identifiers
date: 2026-09-22
status: None
missing fields: []
references: ['2025-08-26-participant-context']


,target,scope,statement
0,2025-08-26-participant-context,partial,the store and service interface for those enti...


## Cell 2.2: One Field

- Each field keeps its Markdown, so later steps can quote it exactly

In [8]:
print(record["fields"]["rationale"][:600])

The [participant context](../2025-08-26-participant-context/README.md) decision record added a `participantContextId`
to these entities, but their ids stayed globally unique and the stores kept resolving them by id alone. In a _virtual_
connector, where several participant contexts share one runtime and one database, this had two consequences:

- **existence oracle**: a participant creating a resource with an id already used by another participant received a
  `409 Conflict`, which reveals the ids in use by other tenants;
- **id squatting**: a participant could reserve ids (for example well-kn


# Part 3: Check a Model's Replacement Claims

## Cell 3.1: Keep Only Claims the Record Supports

- The rule parser only understands EDC's own wording; on six other ADR
  repositories it found 0 of 7 replacements
- With an API key, `relations.py` asks a chat model instead, then keeps a
  claim only if its target is a real record and its evidence quote appears
  word for word in the record
- Here a scripted fake model makes one true claim and two invented ones, so
  we can see the checks work without a key

In [9]:
true_claim = {
    "type": "supersedes",
    "target": "2025-08-26-participant-context",
    "scope": "partial",
    "evidence": "It supersedes the statement of the [participant context]"
    "(../2025-08-26-participant-context/README.md) decision record",
    "replaced": "the store and service interface for those entities won't change",
}
invented_target = {**true_claim, "target": "2023-01-01-made-up-record"}
invented_quote = {
    **true_claim,
    "target": "2025-05-28-dataspace-profile-context",
    "evidence": "This replaces the dataspace profile context record.",
}
chat = utils.ScriptedChat([true_claim, invented_target, invented_quote])
edges = arelati.model_edges(record_id, text, corpus, chat)
display(pd.DataFrame(edges)[["newer", "older", "scope", "statement"]])

,newer,older,scope,statement
0,2026-09-22-per-participant-resource-ids,2025-08-26-participant-context,partial,the store and service interface for those enti...


- Only the true claim survives: the invented target is not in the corpus,
  and the invented quote is not in the record
- In `build`, model edges are merged with the rule edges, so a model miss
  can never remove an edge the rules found

# Part 4: Link the Record to Code

## Cell 4.1: Names in the Text

- `Linker` pulls tokens out of the text and resolves each against the tree:
  - `explicit_class`: a CamelCase name that matches a `.java` file
  - `explicit_module`: a kebab-case name that matches a directory
  - `explicit_path`: a path in backticks
  - `explicit_file`: a file name that is unique in the tree

In [10]:
linker = alinker.Linker(store)
links = linker.link_record(record_id, text)
links_df = pd.DataFrame(links)
display(links_df["signal"].value_counts())
display(links_df[["signal", "evidence", "file_id"]].head(8))

signal
explicit_class    21
Name: count, dtype: int64

,signal,evidence,file_id
0,explicit_class,Asset,spi/control-plane-spi/src/main/java/org/eclips...
1,explicit_class,AssetIndex,spi/control-plane-spi/src/main/java/org/eclips...
2,explicit_class,AssetIndexTestBase,spi/control-plane-spi/src/testFixtures/java/or...
3,explicit_class,AssetService,spi/control-plane-spi/src/main/java/org/eclips...
4,explicit_class,ConsumerOfferResolver,spi/control-plane-spi/src/main/java/org/eclips...
5,explicit_class,ContractAgreement,spi/control-plane-spi/src/main/java/org/eclips...
6,explicit_class,ContractDefinition,spi/control-plane-spi/src/main/java/org/eclips...
7,explicit_class,ContractDefinitionService,spi/control-plane-spi/src/main/java/org/eclips...


## Cell 4.2: Files in the Commit That Added the Record

- The second signal is the commit that added the record: if code changed in
  the same commit, that code likely implements the decision

In [11]:
commit_links, report = linker.adr_commit_links(record_id, record["path"])
print(report)
print("links from the commit signal:", len(commit_links))

{'record_id': '2026-09-22-per-participant-resource-ids', 'commit': 'bb07ccc67a2342c756d3680c8eadbac8971d4644', 'files': 2, 'code_files': 0, 'code_files_still_present': 0}
links from the commit signal: 0


- On EDC this signal finds nothing: those commits touch only the record

# Part 5: Build and Query the Decision Graph

## Cell 5.1: Parse and Link All Ten Records

In [12]:
texts = store.read_many(
    [f"{aconfig.RECORD_ROOT}{r}/README.md" for r in aconfig.RECORDS]
)
text_of = {
    r: texts[f"{aconfig.RECORD_ROOT}{r}/README.md"] for r in aconfig.RECORDS
}
records = [aparser.parse_record(r, text_of[r], corpus) for r in aconfig.RECORDS]
all_links = [
    l for r in aconfig.RECORDS for l in linker.link_record(r, text_of[r])
]
graph = agraph.build_graph(records, all_links)
kinds = pd.Series(
    [d["kind"] for _, _, d in graph.edges(data=True)]
).value_counts()
print("nodes:", graph.number_of_nodes())
display(kinds)

nodes: 120


governs       128
references      7
supersedes      1
Name: count, dtype: int64

## Cell 5.2: Ask the Graph Questions

- Which records govern a directory?
- Which newer record replaces an older one, and which statement?

In [13]:
directory = "spi/control-plane-spi/src/main/java/org/eclipse/edc/protocol/spi"
print("records governing", directory)
for rid in agraph.governing_records(graph, directory):
    print("  ", rid)

records governing spi/control-plane-spi/src/main/java/org/eclipse/edc/protocol/spi
   2025-05-28-dataspace-profile-context
   2025-07-25-multiple-participant-identifiers
   2025-10-29-participant-identifiers
   2026-05-11-multi-profile-virtual-connector


In [14]:
for newer, edge in agraph.superseded_by(graph, "2025-08-26-participant-context"):
    print(f"{newer} replaces part of 2025-08-26-participant-context")
    print("statement no longer true:", edge["statement"])

2026-09-22-per-participant-resource-ids replaces part of 2025-08-26-participant-context
statement no longer true: the store and service interface for those entities won't change


# Part 6: Staleness and Gaps

## Cell 6.1: Is the Record Still Current?

- `stale = later commits to the linked code / (1 + later edits to the record)`
- Shared build files are linked but left out of the commit count, since
  almost every commit touches them
- Names the record uses that no longer exist in the code are reported too

In [15]:
files = sorted({l["file_id"] for l in all_links if l["record_id"] == record_id})
dead = astalen.dead_identifiers(store, text, linker.classes, linker.dirs_by_name)
score = astalen.score_record(store, record, files, dead)
print(score)
threshold = aconfig.get_settings().stale_warn_threshold
for warning in astalen.warnings_for(score, threshold, record):
    print("-", warning["plain"])

{'record_id': '2026-09-22-per-participant-resource-ids', 'edits_after_first': 0, 'n_linked_files': 21, 'dead_identifiers': [], 'stale': 11.0, 'reason': None, 'churn_code': 11}


- No warning: the score is below the default threshold of 15

## Cell 6.2: Complex Code No Record Covers

- `lizard` measures cyclomatic complexity per function; it is summed per
  module and multiplied by the share of the module's files that no record
  links to

In [16]:
governed = {l["file_id"] for l in all_links}
gaps = agaps.rank_gaps(store, governed, top=5)
display(pd.DataFrame(gaps))

,module,complexity,coverage,gap
0,spi/control-plane-spi,1754,0.091,1594.09
1,core/common/lib/core-lib,1188,0.019,1165.44
2,spi/core-spi,1067,0.059,1004.50
3,core/control-plane/lib/control-plane-lib,366,0.000,366.00
4,core/control-plane/control-plane-transform,310,0.000,310.00
